### Phase II
#### Hypothesis: How does the amount of sleep (different hours) affect the average college student in terms of productivity (freshman or senior)?

### Step 0 Install and import pandas
This cell installs pandas if needed and imports it so we can work with the datasets.

In [ ]:
# Install pandas in case it is missing from the environment
%pip install pandas

# Import pandas for loading and cleaning the data
import pandas as pd

### Step 1 Load all three datasets
This cell loads the grades, Fitbit sleep, and CMU sleep files from the repo so we can inspect them side by side.

In [ ]:
# Base URL for the raw CSV files in our GitHub repo
base = "https://raw.githubusercontent.com/samira-maria/2950Project/refs/heads/main/CSVs/"

# Load each dataset into its own dataframe
grades = pd.read_csv(base + "CourseGrades(3-6-20).csv")
fitbit = pd.read_csv(base + "FitbitSleep(1-30-20).csv")
cmu = pd.read_csv(base + "cmu-sleep.csv")

# Print the shape of each so we know how many rows and columns we start with
for name, d in [("grades", grades), ("fitbit", fitbit), ("cmu", cmu)]:
    print(name, d.shape)

### Step 2 Inspect columns and missing values
This cell shows the column names, data types, and missing value counts so we know what needs cleaning.

In [ ]:
# Show column names, types, and non null counts for each dataset
for name, d in [("grades", grades), ("fitbit", fitbit), ("cmu", cmu)]:
    print("=====", name, "=====")
    d.info()
    print(d.head(3), "\n")

### Step 3 Clean the grades data
This cell converts letter grades to a 4.0 scale and drops rows with grades that do not count toward GPA, such as withdrawals.

In [ ]:
# Map letter grades to grade points on a 4.0 scale
grade_points = {"A+": 4.0, "A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7,
                "C+": 2.3, "C": 2.0, "C-": 1.7, "D+": 1.3, "D": 1.0, "D-": 0.7, "F": 0.0}

# Create a numeric column, non letter grades (W, P, etc.) become NaN
grades["grade_points"] = grades["FinalGrade"].map(grade_points)

# Drop rows without a usable numeric grade and remove exact duplicate rows
grades_clean = grades.dropna(subset=["grade_points"]).drop_duplicates()

# Compare row counts before and after cleaning
print(len(grades), "->", len(grades_clean))

### Step 4 Build a term GPA per student
Each student has many course rows, so this cell averages grade points by student and academic period to get one productivity number per student per term.

In [ ]:
# Average grade points per student per term, this is our productivity measure
term_gpa = (grades_clean
            .groupby(["egoid", "AcademicPeriod"], as_index=False)["grade_points"]
            .mean()
            .rename(columns={"grade_points": "term_gpa"}))

# Preview the result
term_gpa.head()

### Step 5 Clean the Fitbit sleep data
This cell removes duplicates and impossible values, then averages nightly sleep per student.

In [ ]:
# Real column names from the Fitbit file
id_col = "egoid"
sleep_col = "minsasleep"

# Drop duplicate rows and rows missing the student id, date, or minutes asleep
fitbit_clean = fitbit.drop_duplicates().dropna(subset=[id_col, "dataDate", sleep_col]).copy()

# Add up all sleep sessions that fall on the same date so each row becomes one night
nightly = fitbit_clean.groupby([id_col, "dataDate"], as_index=False)[sleep_col].sum()

# Convert minutes to hours and drop impossible nights (under 1 hour or over 16 hours)
nightly["sleep_hours"] = nightly[sleep_col] / 60
nightly = nightly[nightly["sleep_hours"].between(1, 16)]

# Average nightly sleep per student
sleep_avg = nightly.groupby(id_col, as_index=False)["sleep_hours"].mean()
sleep_avg.head()

### Step 6 Merge sleep and grades
This cell joins average sleep to term GPA by student id so each row holds one student term with that student's sleep habits.

In [ ]:
# Inner join keeps only students who appear in both datasets
merged = term_gpa.merge(sleep_avg, on="egoid", how="inner")

# Bucket sleep into groups so we can compare different hours of sleep
merged["sleep_group"] = pd.cut(merged["sleep_hours"], bins=[0, 6, 7, 8, 24],
                               labels=["under 6", "6 to 7", "7 to 8", "8 plus"])
merged.head()

### Step 7 Summary statistics
This cell reports averages, spreads, counts, and group comparisons for our key variables.

In [ ]:
# Basic stats (count, mean, std, min, quartiles, max) for the key numeric variables
print(merged[["sleep_hours", "term_gpa"]].describe())

# Average GPA, spread, and number of student terms in each sleep group
print(merged.groupby("sleep_group", observed=True)["term_gpa"].agg(["count", "mean", "median", "std"]))

# Correlation between sleep and GPA
print("Correlation", merged["sleep_hours"].corr(merged["term_gpa"]))

# Totals for context
print("Unique students", merged["egoid"].nunique(), "Student terms", len(merged))

**Freshman versus senior note.** The grades file has no class year column, so to compare freshmen and seniors we need to infer year from each student's earliest AcademicPeriod. The cmu-sleep file has a cohort variable but only includes first year students, so it cannot speak to seniors.

## Data Description

**What do the rows and columns represent?**
Each row in our final merged dataset is one student in one academic term. The columns are the anonymized student id (egoid), the academic period, term GPA, average nightly sleep in hours, and the sleep group the student falls into (under 6, 6 to 7, 7 to 8, or 8 plus hours). Term GPA is the average of a student's final course grades converted to a 4.0 scale. Average nightly sleep comes from Fitbit data, where each night is the total minutes asleep across all sleep sessions on that date. The raw Fitbit file has one row per sleep session with the bed time, time out of bed, time in bed, minutes to fall asleep, minutes asleep, minutes awake, and a sleep efficiency score. The raw grades file has one row per student per course with the student id, academic period, course reference number, and final letter grade. We also load a separate CMU sleep file with 634 students that includes term GPA, cumulative GPA, total sleep time, bedtime variability, midpoint of sleep, cohort, and demographic variables. It uses a different student id (subject_id), so we analyze it on its own.

**What is the source and purpose of each file?**
The grades and Fitbit sleep files come from the NetHealth data release hosted by the University of Notre Dame. The release pairs wearable Fitbit data with registrar records for Notre Dame students and is shared for research use, which makes it well suited to studying how sleep relates to grades. The student id lets us link each student's sleep records to their grades. The CMU sleep file comes from the CMU Statistics and Data Science Data Repository and was created to study whether nightly sleep duration predicts end of semester GPA in first year college students. It combines five studies across Carnegie Mellon, the University of Washington, and Notre Dame, and it is based on the research of Creswell and colleagues published in PNAS in 2023.

**Why was our dataset created?**
We built the merged dataset to test our hypothesis that the amount of sleep a college student gets is related to productivity, which we measure with term GPA. We want to compare how different amounts of sleep relate to academic performance, and to look at differences between freshmen and seniors.

**How was the dataset created?**
We loaded three CSV files from our GitHub repository. For the main analysis we converted letter grades to grade points and averaged them by student and term to get term GPA. We totaled sleep sessions into nightly sleep, averaged those nights per student, and joined the result to term GPA using the student id. We then grouped students by their average hours of sleep so we could compare outcomes across different amounts of sleep.

**What might be missing from the dataset?**
Grades that are not on a letter scale, such as withdrawals and pass fail courses, were dropped, so those courses are not reflected in term GPA. Students who appear in only one of the two files are lost in the merge. Nights where the Fitbit was not worn or did not record are missing, which means a student's average sleep is based only on the nights we can see. The grades file has no class year, so freshman or senior status has to be inferred from each student's earliest academic period. The CMU file only includes first year students, so it cannot speak to seniors. The data also has no information about workload, major, stress, jobs, or other factors that could affect both sleep and grades.

**What are the exact preparation steps?**
1. Load the grades, Fitbit, and CMU files from the repository.
2. Inspect the columns, data types, and missing values in each file.
3. Map letter grades to grade points on a 4.0 scale and drop rows with grades that have no numeric value.
4. Remove duplicate rows from the grades data.
5. Average grade points by student and academic period to create term GPA.
6. Remove duplicate rows from the Fitbit data and drop rows missing the student id, date, or minutes asleep.
7. Add up minutes asleep across all sessions on the same date to get one value per night.
8. Convert minutes to hours and drop nights under 1 hour or over 16 hours as likely recording errors.
9. Average nightly sleep hours for each student.
10. Join average sleep to term GPA on the student id, keeping only students found in both files.
11. Bucket average sleep into four groups for comparison.

**Does the dataset use human data?**
Yes. Every row describes a real student, including their grades, their sleep behavior measured by a wearable device, and in the CMU file their race, gender, and first generation status. Student ids are anonymized, but the records are still human subjects data, so we treat them as sensitive. We use them only for this analysis and we do not try to identify any individual student.

## Data Limitations

Our sample comes from a specific set of students who agreed to wear Fitbits, so it may skew toward students who are healthier, more tech comfortable, or more motivated, and it may not represent all college students. Fitbit sleep tracking is an estimate and can misclassify wake and sleep time. Averaging sleep across all nights hides variation such as all nighters before exams, which may matter more than the average. GPA is an imperfect measure of productivity because it ignores work outside class, course difficulty, and grading differences across majors. Sleep and grades overlap in time unevenly, since sleep was measured in one window and grades span several terms. Finally, any link we find is a correlation. Stress, major, and workload could drive both sleep and grades. If results are misread, they could push students toward oversimplified advice or reinforce unfair assumptions about students who sleep less because of jobs, caregiving, or health, so conclusions should be stated carefully.